# 01 - Ground-truth generation

For each Movebank study in the knowledge base, use an LLM to generate a
small set of realistic user questions whose answer should be that study.
We use this Q -> gold-study-id set for retrieval and LLM evaluations.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from ingest import load_documents
from llmclient import get_llm_client

docs = load_documents()
print(f'{len(docs)} studies')

In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

GEN_INSTRUCTIONS = '''You generate short natural questions a biologist or\ncurious user might ask that a Movebank study record can answer. Focus on:\n- species / taxa\n- location / region\n- sensors used\n- principal investigator\n- number of tracked animals\n- time period.\nReturn 5 questions.'''

def gen_questions(doc, client):
    user = f'''Study: {doc.get("name")}\nSpecies: {doc.get("taxa")}\nLocation: {doc.get("location")}\nPI: {doc.get("principal_investigator")}\nSensors: {doc.get("sensor_types")}\nPeriod: {doc.get("time_period")}\nObjective: {doc.get("study_objective")}'''
    resp = client.beta.chat.completions.parse(
        model=client.model,
        messages=[{'role':'system','content':GEN_INSTRUCTIONS},{'role':'user','content':user}],
        response_format=Questions,
        temperature=0.3,
    )
    return resp.choices[0].message.parsed.questions

client = get_llm_client()

In [ ]:
import json
from tqdm.auto import tqdm

ground_truth = []
for doc in tqdm(docs):
    try:
        for q in gen_questions(doc, client):
            ground_truth.append({'question': q, 'study_id': doc['study_id']})
    except Exception as e:
        print('skip', doc['study_id'], e)

with open('ground_truth.json','w') as f:
    json.dump(ground_truth, f, indent=2)
print(len(ground_truth), 'questions')